In [ ]:
import ollama
from ollama import chat

# Import der Sätze

In [ ]:
import pandas as pd

datafr = pd.read_csv('../data/final.csv')

saetze = datafr["input"].tolist()


In [ ]:



def classify_with_rules(latin_text: str) -> str:
    response = ollama.chat(
        model='augustulus-latin',
        messages=[{'role': 'user', 'content': latin_text}]
    )
    prediction = response.message.content.strip()
    
    
    text_lower = latin_text.lower()
    
    extreme_neg = ['crudel', 'saev', 'trucidat', 'perdi', 'desperatio']
    extreme_pos = ['splendidissim', 'magnificus', 'beatitudo', 'triumphus magnificus']
    
    has_extreme_neg = any(m in text_lower for m in extreme_neg)
    has_extreme_pos = any(m in text_lower for m in extreme_pos)
    exclamations   = latin_text.count('!')
    
    if 'MODERATELY NEGATIVE' in prediction and has_extreme_neg:
        return 'VERY NEGATIVE'
    if 'MODERATELY POSITIVE' in prediction and has_extreme_pos and exclamations >= 2:
        return 'EXTREMELY POSITIVE'
    
    return prediction

# For-Loop mit Post-Processing
results = []
for satz in saetze:
    results.append({
        'input':      satz,
        'output':     classify_with_rules(satz),
    })

df = pd.DataFrame(results)

In [ ]:
response = ollama.chat(
        model='augustulus-latin',
        messages=[{'role': 'user', 'content': "Victoria splendidissima! Dux gloriam aeternam meruit!"}]
    )
prediction = response.message.content.strip()
prediction


In [ ]:
classify_with_rules("Victoria splendidissima! Dux gloriam aeternam meruit! ")

In [ ]:
classify_with_rules("Bellum crudele et longum populum afflixerat.")

In [ ]:
classify_with_rules("ut enim Aristarchus Homeri versum negat, quem non proba sic tu - libet enim mihi iocari - , quod disertum non erit, ne putaris meum.")

In [ ]:
# CSV speichern
df.to_csv('../data/sentiment.csv', index=False, encoding='utf-8')